# Scene Classification

It includes:

1. Clean dataset pipeline with corrupt image handling.
2. Baseline CNN from scratch.
3. Deeper CNN with regularization.
4. Optimizer comparison: Adam vs SGD.
5. Ablation study: removing Dropout.
6. Transfer learning using MobileNetV2.
7. Optional ResNet50 comparison if GPU time allows.
8. Final model comparison, model selection, error analysis, and report-ready outputs.


## 1. Imports and Reproducibility

Importing libraries required for data loading, training, evaluation, and visualization.

In [6]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [7]:
# Checking for GPU

print("GPU devices:", tf.config.list_physical_devices("GPU"))
!nvidia-smi

GPU devices: []
/bin/bash: line 1: nvidia-smi: command not found


## 2. Mount Google Drive and Set Paths




In [8]:
from google.colab import drive
drive.mount("/content/drive")

ValueError: Mountpoint must not already contain files

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/AI_ML/CourseWork/Scene Classification")

BASE_DIR = DATASET_DIR
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"

# Save all generated models, CSVs, and summaries beside the dataset folder.
PROJECT_DIR = BASE_DIR.parent
OUTPUT_DIR = PROJECT_DIR / "Scene Classification Outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory exists:", BASE_DIR.exists())
print("Train directory exists:", TRAIN_DIR.exists())
print("Test directory exists:", TEST_DIR.exists())
print("Output directory exists:", OUTPUT_DIR.exists())
print("Output directory:", OUTPUT_DIR)

if not BASE_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {BASE_DIR}")

if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"Train folder not found: {TRAIN_DIR}")

if not TEST_DIR.exists():
    raise FileNotError(f"Test folder not found: {TEST_DIR}")

## 3. Class Verification

The class names are detected from the training folder and checked against the test folder.

In [ ]:
train_classes = sorted([folder.name for folder in TRAIN_DIR.iterdir() if folder.is_dir()])
test_classes = sorted([folder.name for folder in TEST_DIR.iterdir() if folder.is_dir()])

if train_classes != test_classes:
    raise ValueError("Train and test class folders do not match. Check dataset structure.")

CLASS_NAMES = train_classes
NUM_CLASSES = len(CLASS_NAMES)

class_to_index = {class_name: index for index, class_name in enumerate(CLASS_NAMES)}
index_to_class = {index: class_name for class_name, index in class_to_index.items()}

print("Classes:", CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)
print("Class mapping:", class_to_index)

## 4. Clean Metadata Creation

This step detects unreadable or corrupt images. Files are not deleted from Google Drive. Invalid files are simply excluded from the training pipeline.

To save time on repeated runs, the notebook reuses `clean_metadata.csv` if it already exists in the output folder.

In [ ]:
clean_metadata_path = OUTPUT_DIR / "clean_metadata.csv"


def scan_images(directory, split_name):
    records = []
    image_extensions = [".jpg", ".jpeg", ".png"]

    for class_folder in sorted(directory.iterdir()):
        if not class_folder.is_dir():
            continue

        class_name = class_folder.name

        for img_path in class_folder.rglob("*"):
            if img_path.suffix.lower() not in image_extensions:
                continue

            try:
                with Image.open(img_path) as img:
                    img.verify()

                with Image.open(img_path) as img:
                    width, height = img.size
                    mode = img.mode

                records.append({
                    "path": str(img_path),
                    "split": split_name,
                    "class": class_name,
                    "width": width,
                    "height": height,
                    "mode": mode,
                    "is_valid": True,
                    "error": None
                })

            except Exception as error:
                records.append({
                    "path": str(img_path),
                    "split": split_name,
                    "class": class_name,
                    "width": np.nan,
                    "height": np.nan,
                    "mode": None,
                    "is_valid": False,
                    "error": str(error)
                })

    return pd.DataFrame(records)


if clean_metadata_path.exists():
    clean_df = pd.read_csv(clean_metadata_path)
    print("Loaded existing clean metadata:", clean_metadata_path)

    # Reconstruct a lightweight meta_df for summary consistency.
    meta_df = clean_df.copy()
    corrupt_df = pd.DataFrame(columns=clean_df.columns)

else:
    train_meta = scan_images(TRAIN_DIR, "train")
    test_meta = scan_images(TEST_DIR, "test")
    meta_df = pd.concat([train_meta, test_meta], ignore_index=True)

    corrupt_df = meta_df[meta_df["is_valid"] == False].copy()
    clean_df = meta_df[meta_df["is_valid"] == True].copy()

    clean_df.to_csv(clean_metadata_path, index=False)
    print("Created and saved clean metadata:", clean_metadata_path)

print("Total valid images:", len(clean_df))
print("Corrupt images found in this run:", len(corrupt_df))

display(clean_df.groupby(["split", "class"]).size().reset_index(name="image_count"))

display(corrupt_df.head())

## 5. Dataset Split and TensorFlow Data Pipeline

The test set remains untouched. The original training set is split into 80% training and 20% validation using stratification.

Two image sizes are prepared:

- `150x150` for scratch CNN models.
- `224x224` for future transfer learning models such as MobileNetV2 or ResNet50.

In [ ]:
SCRATCH_IMG_SIZE = (150, 150)
TRANSFER_IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VALIDATION_SIZE = 0.20

train_clean_df = clean_df[clean_df["split"] == "train"].copy()
test_clean_df = clean_df[clean_df["split"] == "test"].copy()

train_clean_df["label"] = train_clean_df["class"].map(class_to_index)
test_clean_df["label"] = test_clean_df["class"].map(class_to_index)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_clean_df["path"].values,
    train_clean_df["label"].values,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=train_clean_df["label"].values
)

test_paths = test_clean_df["path"].values
test_labels = test_clean_df["label"].values

print("Training samples:", len(train_paths))
print("Validation samples:", len(val_paths))
print("Testing samples:", len(test_paths))

In [ ]:
def load_image(path, label, image_size):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, image_size)

    label = tf.one_hot(label, depth=NUM_CLASSES)
    return image, label


def create_dataset(paths, labels, image_size, batch_size=BATCH_SIZE, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)

    dataset = dataset.map(
        lambda path, label: load_image(path, label, image_size),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_ds_150 = create_dataset(train_paths, train_labels, SCRATCH_IMG_SIZE, shuffle=True)
val_ds_150 = create_dataset(val_paths, val_labels, SCRATCH_IMG_SIZE, shuffle=False)
test_ds_150 = create_dataset(test_paths, test_labels, SCRATCH_IMG_SIZE, shuffle=False)

train_ds_224 = create_dataset(train_paths, train_labels, TRANSFER_IMG_SIZE, shuffle=True)
val_ds_224 = create_dataset(val_paths, val_labels, TRANSFER_IMG_SIZE, shuffle=False)
test_ds_224 = create_dataset(test_paths, test_labels, TRANSFER_IMG_SIZE, shuffle=False)

for images, labels in train_ds_150.take(1):
    print("150x150 image batch:", images.shape)
    print("Label batch:", labels.shape)
    print("Pixel range before normalization:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

for images, labels in train_ds_224.take(1):
    print("224x224 image batch:", images.shape)

## 6. Preprocessing Layers

Data augmentation is used only during training. Normalization rescales pixel values from `0-255` to `0-1`.

In [ ]:
def get_preprocessing_layers():
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10)
    ], name="data_augmentation")

    normalization_layer = layers.Rescaling(1./255, name="normalization")
    return data_augmentation, normalization_layer


def show_sample_images(dataset, title="Sample Images"):
    plt.figure(figsize=(10, 10))

    for images, labels in dataset.take(1):
        for i in range(9):
            label_index = int(np.argmax(labels[i].numpy()))
            plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(index_to_class[label_index])
            plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


show_sample_images(train_ds_150, "Clean Training Images")

## EDA: Exploratory Data Analysis

This section fulfils the assignment requirement for thorough dataset understanding before modelling begins. All analysis uses `clean_df`, the validated metadata table built in Section 4.

Tasks covered:
1. Dataset overview (total images, classes, split sizes).
2. Class distribution — bar chart and percentage table.
3. Split distribution across classes — stacked bar chart.
4. Image dimension statistics — width/height distributions.
5. Aspect-ratio analysis.
6. Colour-mode breakdown.
7. Sample images per class.
8. Data-augmentation preview.

In [ ]:
# ============================================================
# EDA 1 — DATASET OVERVIEW
# ============================================================

total_images = len(clean_df)
train_count  = len(clean_df[clean_df["split"] == "train"])
test_count   = len(clean_df[clean_df["split"] == "test"])

print("=" * 50)
print(f"Dataset: Intel Scene Classification")
print(f"Total valid images  : {total_images:,}")
print(f"Training images     : {train_count:,}")
print(f"Test images         : {test_count:,}")
print(f"Number of classes   : {NUM_CLASSES}")
print(f"Classes             : {', '.join(CLASS_NAMES)}")
print("=" * 50)

# Per-split, per-class counts
overview_df = (
    clean_df
    .groupby(["split", "class"])
    .size()
    .reset_index(name="count")
    .pivot(index="class", columns="split", values="count")
    .fillna(0)
    .astype(int)
)
overview_df["total"] = overview_df.sum(axis=1)
overview_df = overview_df.sort_values("total", ascending=False)

print("\nImages per class and split:")
display(overview_df)


In [ ]:
# ============================================================
# EDA 2 — CLASS DISTRIBUTION BAR CHART
# ============================================================

train_class_counts = (
    clean_df[clean_df["split"] == "train"]
    .groupby("class")
    .size()
    .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(train_class_counts.index, train_class_counts.values, color="steelblue", edgecolor="black")
axes[0].set_title("Training Images per Class", fontsize=13)
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Image Count")
axes[0].tick_params(axis="x", rotation=30)
for idx, val in enumerate(train_class_counts.values):
    axes[0].text(idx, val + 10, str(val), ha="center", va="bottom", fontsize=9)

# Pie chart — proportion
axes[1].pie(
    train_class_counts.values,
    labels=train_class_counts.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Set3.colors[:NUM_CLASSES]
)
axes[1].set_title("Class Proportion (Training Split)", fontsize=13)

plt.suptitle("Class Distribution Analysis", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

# Print percentage table
pct_df = pd.DataFrame({
    "Class": train_class_counts.index,
    "Count": train_class_counts.values,
    "Percentage (%)": (train_class_counts.values / train_class_counts.sum() * 100).round(2)
})
print("\nClass distribution summary:")
display(pct_df.reset_index(drop=True))

is_balanced = train_class_counts.max() / train_class_counts.min() < 1.5
print(f"\nDataset balance assessment: {'Balanced' if is_balanced else 'Imbalanced'}")
print(f"Max class count: {train_class_counts.max()}, Min class count: {train_class_counts.min()}")


In [ ]:
# ============================================================
# EDA 3 — TRAIN vs TEST SPLIT PER CLASS (STACKED BAR)
# ============================================================

train_counts = (
    clean_df[clean_df["split"] == "train"]
    .groupby("class").size().reindex(CLASS_NAMES, fill_value=0)
)
test_counts = (
    clean_df[clean_df["split"] == "test"]
    .groupby("class").size().reindex(CLASS_NAMES, fill_value=0)
)

x = np.arange(NUM_CLASSES)
width = 0.6

fig, ax = plt.subplots(figsize=(12, 5))
bars_train = ax.bar(x, train_counts.values, width, label="Train", color="steelblue", edgecolor="black")
bars_test  = ax.bar(x, test_counts.values, width, bottom=train_counts.values, label="Test", color="coral", edgecolor="black")

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_ylabel("Image Count")
ax.set_title("Train / Test Split per Class", fontsize=13)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# EDA 4 — IMAGE DIMENSION STATISTICS
# ============================================================

dim_df = clean_df[clean_df["split"] == "train"].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Width histogram
axes[0].hist(dim_df["width"], bins=30, color="steelblue", edgecolor="black")
axes[0].set_title("Image Width Distribution (Train)")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Count")
axes[0].axvline(dim_df["width"].median(), color="red", linestyle="--", label=f"Median={dim_df['width'].median():.0f}")
axes[0].legend()

# Height histogram
axes[1].hist(dim_df["height"], bins=30, color="darkorange", edgecolor="black")
axes[1].set_title("Image Height Distribution (Train)")
axes[1].set_xlabel("Height (px)")
axes[1].set_ylabel("Count")
axes[1].axvline(dim_df["height"].median(), color="red", linestyle="--", label=f"Median={dim_df['height'].median():.0f}")
axes[1].legend()

# Scatter: width vs height
axes[2].scatter(dim_df["width"], dim_df["height"], alpha=0.3, s=5, color="purple")
axes[2].set_title("Width vs Height")
axes[2].set_xlabel("Width (px)")
axes[2].set_ylabel("Height (px)")

plt.suptitle("Image Dimension Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary statistics
print("\nDimension statistics (training set):")
print(dim_df[["width", "height"]].describe().round(1))
print(f"\nAll images same size: {(dim_df['width'].nunique() == 1 and dim_df['height'].nunique() == 1)}")


In [ ]:
# ============================================================
# EDA 5 — ASPECT RATIO ANALYSIS
# ============================================================

dim_df = clean_df[clean_df["split"] == "train"].copy()
dim_df["aspect_ratio"] = dim_df["width"] / dim_df["height"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(dim_df["aspect_ratio"], bins=30, color="teal", edgecolor="black")
axes[0].axvline(1.0, color="red", linestyle="--", label="Square (1:1)")
axes[0].set_title("Aspect Ratio Distribution (Train)")
axes[0].set_xlabel("Width / Height")
axes[0].set_ylabel("Count")
axes[0].legend()

aspect_counts = dim_df["aspect_ratio"].round(2).value_counts().head(10)
axes[1].barh(aspect_counts.index.astype(str), aspect_counts.values, color="teal", edgecolor="black")
axes[1].set_title("Top 10 Aspect Ratios")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Aspect Ratio")

plt.tight_layout()
plt.show()

print("\nAspect ratio summary:")
print(dim_df["aspect_ratio"].describe().round(3))


In [ ]:
# ============================================================
# EDA 6 — COLOUR MODE BREAKDOWN
# ============================================================

mode_counts = clean_df["mode"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(mode_counts.index, mode_counts.values, color="mediumseagreen", edgecolor="black")
ax.set_title("Image Colour Mode Distribution")
ax.set_xlabel("Mode")
ax.set_ylabel("Count")
for i, val in enumerate(mode_counts.values):
    ax.text(i, val + 5, str(val), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

print("\nColour mode counts:")
display(mode_counts.reset_index().rename(columns={"index": "mode", "mode": "count"}))


In [ ]:
# ============================================================
# EDA 7 — SAMPLE IMAGES PER CLASS (GRID)
# ============================================================

SAMPLES_PER_CLASS = 4

fig, axes = plt.subplots(NUM_CLASSES, SAMPLES_PER_CLASS, figsize=(SAMPLES_PER_CLASS * 3, NUM_CLASSES * 3))

for row_idx, class_name in enumerate(CLASS_NAMES):
    class_paths = (
        clean_df[(clean_df["split"] == "train") & (clean_df["class"] == class_name)]
        ["path"]
        .sample(n=SAMPLES_PER_CLASS, random_state=SEED)
        .tolist()
    )

    for col_idx, img_path in enumerate(class_paths):
        img = Image.open(img_path).convert("RGB")
        ax = axes[row_idx][col_idx]
        ax.imshow(img)
        ax.axis("off")
        if col_idx == 0:
            ax.set_ylabel(class_name, fontsize=10, rotation=0, labelpad=60, va="center")

plt.suptitle("Sample Images per Class (Training Set)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# EDA 8 — AUGMENTATION PREVIEW (as required by guidelines)
# ============================================================

# This shows how augmentation affects training images before feeding to the model.
data_augmentation, _ = get_preprocessing_layers()

train_class_paths = (
    clean_df[(clean_df["split"] == "train") & (clean_df["class"] == CLASS_NAMES[0])]
    ["path"]
    .sample(n=1, random_state=SEED)
    .values
)

sample_image = tf.io.read_file(train_class_paths[0])
sample_image = tf.io.decode_image(sample_image, channels=3, expand_animations=False)
sample_image.set_shape([None, None, 3])
sample_image = tf.image.resize(sample_image, SCRATCH_IMG_SIZE)

# Expand dims for batch dimension expected by augmentation layer
sample_batch = tf.expand_dims(sample_image, 0)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

# Original
axes[0][0].imshow(sample_image.numpy().astype("uint8"))
axes[0][0].set_title("Original", fontweight="bold")
axes[0][0].axis("off")

# 9 augmented variants
for i in range(9):
    row, col = divmod(i + 1, 5)
    augmented = data_augmentation(sample_batch, training=True)[0]
    axes[row][col].imshow(augmented.numpy().astype("uint8"))
    axes[row][col].set_title(f"Augmented #{i+1}")
    axes[row][col].axis("off")

plt.suptitle(
    f"Data Augmentation Preview — Class: {CLASS_NAMES[0].capitalize()}\n"
    "(Random flip, rotation, zoom, contrast)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()


## 7. Reusable Training, Evaluation, Plotting, and Saving Functions

These helper functions prevent repeated code in Day 3, Day 4, and Day 5.

In [ ]:
def get_callbacks(model_name, patience=5):
    checkpoint_path = OUTPUT_DIR / f"{model_name}_best.keras"

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=max(2, patience // 2),
            min_lr=1e-6
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_loss",
            save_best_only=True
        )
    ]
    return callbacks


def get_optimizer(name="adam", learning_rate=0.001):
    name = name.lower()

    if name == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate)

    if name == "sgd":
        return keras.optimizers.SGD(
            learning_rate=learning_rate,
            momentum=0.9,
            nesterov=True
        )

    raise ValueError("Supported optimizers: 'adam' or 'sgd'.")


def compile_model(model, optimizer_name="adam", learning_rate=0.001):
    model.compile(
        optimizer=get_optimizer(optimizer_name, learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def train_model(model, train_ds, val_ds, model_name, epochs, optimizer_name="adam", learning_rate=0.001, patience=5):
    model = compile_model(model, optimizer_name=optimizer_name, learning_rate=learning_rate)

    start_time = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=get_callbacks(model_name, patience=patience)
    )
    training_time = time.time() - start_time

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(OUTPUT_DIR / f"{model_name}_history.csv", index=False)

    return model, history_df, training_time


def plot_history(history_df, title_prefix):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["accuracy"], label="Training Accuracy")
    plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{title_prefix}: Training vs Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["loss"], label="Training Loss")
    plt.plot(history_df["val_loss"], label="Validation Loss")
    plt.title(f"{title_prefix}: Training vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


def collect_predictions(model, dataset):
    y_true = []
    y_pred = []

    for images, labels in dataset:
        predictions = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(predictions, axis=1))

    return np.array(y_true), np.array(y_pred)


def evaluate_model(model, dataset, model_name):
    test_loss, test_accuracy = model.evaluate(dataset, verbose=1)
    y_true, y_pred = collect_predictions(model, dataset)

    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(OUTPUT_DIR / f"{model_name}_classification_report.csv")

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(9, 9))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, values_format="d")
    plt.title(f"{model_name}: Confusion Matrix")
    plt.xticks(rotation=45)
    plt.show()

    metrics = {
        "model": model_name,
        "test_accuracy": float(test_accuracy),
        "test_loss": float(test_loss),
        "macro_f1": float(report_df.loc["macro avg", "f1-score"]),
        "weighted_f1": float(report_df.loc["weighted avg", "f1-score"])
    }

    return metrics, report_df, cm


def plot_sample_predictions(model, dataset, model_name, number=9):
    plt.figure(figsize=(12, 12))

    for images, labels in dataset.take(1):
        predictions = model.predict(images, verbose=0)

        for i in range(min(number, len(images))):
            true_index = int(np.argmax(labels[i].numpy()))
            pred_index = int(np.argmax(predictions[i]))
            confidence = float(np.max(predictions[i]) * 100)

            true_label = index_to_class[true_index]
            pred_label = index_to_class[pred_index]

            plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)")
            plt.axis("off")

    plt.suptitle(f"{model_name}: Sample Predictions", fontsize=16)
    plt.tight_layout()
    plt.show()


def save_model(model, model_name):
    model_path = OUTPUT_DIR / f"{model_name}.keras"
    model.save(model_path)
    print("Saved model to:", model_path)


def add_result(results, model, metrics, training_time_seconds):
    results.append({
        "Model": metrics["model"],
        "Test Accuracy": metrics["test_accuracy"],
        "Test Loss": metrics["test_loss"],
        "Macro F1": metrics["macro_f1"],
        "Weighted F1": metrics["weighted_f1"],
        "Training Time (minutes)": training_time_seconds / 60,
        "Parameters": model.count_params()
    })
    return results


def plot_metric_comparison(comparison_df, metric, title):
    plt.figure(figsize=(8, 5))
    plt.bar(comparison_df["Model"], comparison_df[metric])
    plt.title(title)
    plt.ylabel(metric)
    plt.xticks(rotation=30, ha="right")
    if "Accuracy" in metric or "F1" in metric:
        plt.ylim(0, 1)
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()

## 8. Model Builder Functions

The baseline model follows the following structure: three convolutional layers, three pooling layers, three fully connected layers, and a softmax output layer.

The deeper model approximately doubles the convolutional depth and adds regularization. `GlobalAveragePooling2D` is used in the deeper model to reduce parameters and overfitting compared with a large Flatten layer.

In [ ]:
def build_baseline_cnn(input_shape=(150, 150, 3), num_classes=NUM_CLASSES):
    data_augmentation, normalization_layer = get_preprocessing_layers()

    model = keras.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,
        normalization_layer,

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),

        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),

        layers.Dense(num_classes, activation="softmax")
    ], name="Baseline_CNN")

    return model


def conv_block(model, filters, use_batch_norm=True, use_dropout=True, dropout_rate=0.25, l2_value=0.001):
    model.add(layers.Conv2D(
        filters, (3, 3), padding="same", activation="relu",
        kernel_regularizer=regularizers.l2(l2_value)
    ))
    if use_batch_norm:
        model.add(layers.BatchNormalization())

    model.add(layers.Conv2D(
        filters, (3, 3), padding="same", activation="relu",
        kernel_regularizer=regularizers.l2(l2_value)
    ))
    if use_batch_norm:
        model.add(layers.BatchNormalization())

    model.add(layers.MaxPooling2D((2, 2)))
    if use_dropout:
        model.add(layers.Dropout(dropout_rate))

    return model


def build_deeper_cnn(
    input_shape=(150, 150, 3),
    num_classes=NUM_CLASSES,
    use_batch_norm=True,
    use_dropout=True,
    l2_value=0.001,
    model_name="Deeper_CNN"
):
    data_augmentation, normalization_layer = get_preprocessing_layers()

    model = keras.Sequential(name=model_name)
    model.add(layers.Input(shape=input_shape))
    model.add(data_augmentation)
    model.add(normalization_layer)

    model = conv_block(model, 32, use_batch_norm, use_dropout, 0.20, l2_value)
    model = conv_block(model, 64, use_batch_norm, use_dropout, 0.25, l2_value)
    model = conv_block(model, 128, use_batch_norm, use_dropout, 0.30, l2_value)

    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(
        256, activation="relu",
        kernel_regularizer=regularizers.l2(l2_value)
    ))
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    if use_dropout:
        model.add(layers.Dropout(0.50))

    model.add(layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(l2_value)
    ))
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    if use_dropout:
        model.add(layers.Dropout(0.40))

    model.add(layers.Dense(64, activation="relu"))
    model.add(layers.Dense(num_classes, activation="softmax"))

    return model

## 9. Training Configuration

These values are chosen to balance quality and runtime in Colab. Early stopping may stop training before the maximum epoch count.

In [ ]:
BASELINE_EPOCHS = 15
DEEPER_EPOCHS = 20
SGD_EPOCHS = DEEPER_EPOCHS
ABLATION_EPOCHS = DEEPER_EPOCHS

results = []

print("Baseline epochs:", BASELINE_EPOCHS)
print("Deeper epochs:", DEEPER_EPOCHS)
print("SGD experiment epochs:", SGD_EPOCHS)
print("Ablation epochs:", ABLATION_EPOCHS)

# Day 3: Baseline CNN From Scratch

Purpose: establish a reference model before adding deeper layers and regularization.

In [ ]:
# ============================================================
# DAY 3: BUILD / RESUME BASELINE CNN
# ============================================================

baseline_final    = OUTPUT_DIR / "baseline_cnn_final.keras"
baseline_history_path = OUTPUT_DIR / "baseline_cnn_history.csv"

if baseline_final.exists():
    print("Baseline final model found. Loading from Drive...")
    baseline_model       = keras.models.load_model(baseline_final)
    baseline_history_df  = pd.read_csv(baseline_history_path)
    baseline_training_time = 0
else:
    print("Building Baseline CNN from scratch...")
    baseline_model = build_baseline_cnn()

baseline_model.summary()


In [ ]:
baseline_model, baseline_history_df, baseline_training_time = train_model(
    model=baseline_model,
    train_ds=train_ds_150,
    val_ds=val_ds_150,
    model_name="baseline_cnn",
    epochs=BASELINE_EPOCHS,
    optimizer_name="adam",
    learning_rate=0.001,
    patience=5
)

print("Baseline training time minutes:", round(baseline_training_time / 60, 2))
plot_history(baseline_history_df, "Baseline CNN")

In [ ]:
# Testing the baseline CNN on test_ds_150

baseline_metrics, baseline_report_df, baseline_cm = evaluate_model(
    baseline_model,
    test_ds_150,
    "baseline_cnn"
)

plot_sample_predictions(baseline_model, test_ds_150, "Baseline CNN")
save_model(baseline_model, "baseline_cnn_final")
results = add_result(results, baseline_model, baseline_metrics, baseline_training_time)

# Day 4: Deeper CNN With Regularization

Purpose: increase model capacity and control overfitting using Batch Normalization, Dropout, L2 regularization, EarlyStopping, and ReduceLROnPlateau.

In [ ]:
checkpoint_path = OUTPUT_DIR / "deeper_cnn_adam_best.keras"

if checkpoint_path.exists():
    print("Checkpoint found. Loading weights from Drive...")
    deeper_model = keras.models.load_model(checkpoint_path)
else:
    print("No checkpoint. Building from scratch...")
    deeper_model = build_deeper_cnn(
        use_batch_norm=True,
        use_dropout=True,
        l2_value=0.001,
        model_name="Deeper_CNN_Adam"
    )

deeper_model, deeper_history_df, deeper_training_time = train_model(
    model=deeper_model,
    train_ds=train_ds_150,
    val_ds=val_ds_150,
    model_name="deeper_cnn_adam",
    epochs=DEEPER_EPOCHS,
    optimizer_name="adam",
    learning_rate=0.001,
    patience=5
)

print("Deeper CNN training time minutes:", round(deeper_training_time / 60, 2))
plot_history(deeper_history_df, "Deeper CNN with Adam")

In [ ]:
deeper_metrics, deeper_report_df, deeper_cm = evaluate_model(
    deeper_model,
    test_ds_150,
    "deeper_cnn_adam"
)

plot_sample_predictions(deeper_model, test_ds_150, "Deeper CNN with Adam")
save_model(deeper_model, "deeper_cnn_adam_final")
results = add_result(results, deeper_model, deeper_metrics, deeper_training_time)

## Baseline vs Deeper CNN Comparison

This comparison addresses model complexity, accuracy, loss, F1-score, parameter count, and training time.

In [ ]:
comparison_df = pd.DataFrame(results)
display(comparison_df)

plot_metric_comparison(comparison_df, "Test Accuracy", "Baseline vs Deeper CNN: Test Accuracy")
plot_metric_comparison(comparison_df, "Test Loss", "Baseline vs Deeper CNN: Test Loss")
plot_metric_comparison(comparison_df, "Training Time (minutes)", "Baseline vs Deeper CNN: Training Time")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(baseline_history_df["val_accuracy"], label="Baseline Validation Accuracy")
plt.plot(deeper_history_df["val_accuracy"], label="Deeper Validation Accuracy")
plt.title("Validation Accuracy: Baseline vs Deeper CNN")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(baseline_history_df["val_loss"], label="Baseline Validation Loss")
plt.plot(deeper_history_df["val_loss"], label="Deeper Validation Loss")
plt.title("Validation Loss: Baseline vs Deeper CNN")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# Day 5: Optimizer Comparison and Ablation Study

Day 5 is designed to show controlled experimental thinking.

1. **Optimizer comparison:** The regularized deeper CNN trained with Adam is compared against the same architecture trained with SGD + momentum.
2. **Ablation study:** Dropout is removed from the deeper CNN to observe whether overfitting increases or generalization decreases.

To save time, the Adam result from Day 4 is reused. Only the SGD and ablation models are trained here.

## 10. Optimizer Comparison: Adam vs SGD

The same deeper architecture is used. Only the optimizer changes.

In [ ]:
sgd_final = OUTPUT_DIR / "deeper_cnn_sgd_final.keras"
sgd_history_path = OUTPUT_DIR / "deeper_cnn_sgd_history.csv"
sgd_checkpoint = OUTPUT_DIR / "deeper_cnn_sgd_best.keras"

if sgd_final.exists():
    print("SGD final model already trained. Loading...")
    sgd_model = keras.models.load_model(sgd_final)
    sgd_history_df = pd.read_csv(sgd_history_path)
    sgd_training_time = 0

elif sgd_checkpoint.exists():
    print("SGD checkpoint found. Loading and continuing training...")
    sgd_model = keras.models.load_model(sgd_checkpoint)

    sgd_model, sgd_history_df, sgd_training_time = train_model(
        model=sgd_model,
        train_ds=train_ds_150,
        val_ds=val_ds_150,
        model_name="deeper_cnn_sgd",
        epochs=DEEPER_EPOCHS,
        optimizer_name="sgd",
        learning_rate=0.001,
        patience=5
    )

else:
    print("Building SGD model from scratch...")
    sgd_model = build_deeper_cnn(
        use_batch_norm=True,
        use_dropout=True,
        l2_value=0.001,
        model_name="Deeper_CNN_SGD"
    )

    sgd_model, sgd_history_df, sgd_training_time = train_model(
        model=sgd_model,
        train_ds=train_ds_150,
        val_ds=val_ds_150,
        model_name="deeper_cnn_sgd",
        epochs=DEEPER_EPOCHS,
        optimizer_name="sgd",
        learning_rate=0.001,
        patience=5
    )

sgd_metrics, sgd_report_df, sgd_cm = evaluate_model(
    sgd_model,
    test_ds_150,
    "deeper_cnn_sgd"
)

save_model(sgd_model, "deeper_cnn_sgd_final")
results = add_result(results, sgd_model, sgd_metrics, sgd_training_time)

In [ ]:
optimizer_comparison_df = pd.DataFrame([
    results[1],  # deeper_cnn_adam
    results[2]   # deeper_cnn_sgd
])

display(optimizer_comparison_df)

plot_metric_comparison(optimizer_comparison_df, "Test Accuracy", "Optimizer Comparison: Adam vs SGD")
plot_metric_comparison(optimizer_comparison_df, "Test Loss", "Optimizer Comparison: Test Loss")
plot_metric_comparison(optimizer_comparison_df, "Training Time (minutes)", "Optimizer Comparison: Training Time")

plt.figure(figsize=(8, 5))
plt.plot(deeper_history_df["val_accuracy"], label="Adam Validation Accuracy")
plt.plot(sgd_history_df["val_accuracy"], label="SGD Validation Accuracy")
plt.title("Optimizer Comparison: Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(deeper_history_df["val_loss"], label="Adam Validation Loss")
plt.plot(sgd_history_df["val_loss"], label="SGD Validation Loss")
plt.title("Optimizer Comparison: Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## 11. Ablation Study: Removing Dropout

This model keeps the same deeper CNN structure, Batch Normalization, and L2 regularization, but removes Dropout. This isolates the contribution of Dropout to generalization.

In [ ]:
ablation_model = build_deeper_cnn(
    use_batch_norm=True,
    use_dropout=False,
    l2_value=0.001,
    model_name="Ablation_No_Dropout"
)

ablation_model.summary()

In [ ]:
ablation_checkpoint = OUTPUT_DIR / "ablation_no_dropout_best.keras"
ablation_final = OUTPUT_DIR / "ablation_no_dropout_final.keras"
ablation_history_path = OUTPUT_DIR / "ablation_no_dropout_history.csv"

if ablation_final.exists():
    print("Ablation final model already trained. Loading...")
    ablation_model = keras.models.load_model(ablation_final)

    if ablation_history_path.exists():
        ablation_history_df = pd.read_csv(ablation_history_path)
    else:
        ablation_history_df = None

    ablation_training_time = 0

elif ablation_checkpoint.exists():
    print("Ablation checkpoint found. Loading and continuing training...")
    ablation_model = keras.models.load_model(ablation_checkpoint)

    ablation_model, ablation_history_df, ablation_training_time = train_model(
        model=ablation_model,
        train_ds=train_ds_150,
        val_ds=val_ds_150,
        model_name="ablation_no_dropout",
        epochs=DEEPER_EPOCHS,
        optimizer_name="adam",
        learning_rate=0.001,
        patience=5
    )

else:
    print("Building ablation model from scratch...")
    ablation_model = build_deeper_cnn(
        use_batch_norm=True,
        use_dropout=False,  # key difference: dropout removed
        l2_value=0.001,
        model_name="Ablation_No_Dropout"
    )

    ablation_model, ablation_history_df, ablation_training_time = train_model(
        model=ablation_model,
        train_ds=train_ds_150,
        val_ds=val_ds_150,
        model_name="ablation_no_dropout",
        epochs=DEEPER_EPOCHS,
        optimizer_name="adam",
        learning_rate=0.001,
        patience=5
    )

ablation_model.summary()

ablation_metrics, ablation_report_df, ablation_cm = evaluate_model(
    ablation_model,
    test_ds_150,
    "ablation_no_dropout"
)

save_model(ablation_model, "ablation_no_dropout_final")
results = add_result(results, ablation_model, ablation_metrics, ablation_training_time)


In [ ]:
ablation_comparison_df = pd.DataFrame([
    results[1],  # deeper_cnn_adam with dropout
    results[3]   # ablation without dropout
])

display(ablation_comparison_df)

plot_metric_comparison(ablation_comparison_df, "Test Accuracy", "Ablation Study: With Dropout vs No Dropout")
plot_metric_comparison(ablation_comparison_df, "Test Loss", "Ablation Study: Test Loss")
plot_metric_comparison(ablation_comparison_df, "Training Time (minutes)", "Ablation Study: Training Time")

plt.figure(figsize=(8, 5))
plt.plot(deeper_history_df["val_accuracy"], label="With Dropout Validation Accuracy")
plt.plot(ablation_history_df["val_accuracy"], label="No Dropout Validation Accuracy")
plt.title("Ablation Study: Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(deeper_history_df["val_loss"], label="With Dropout Validation Loss")
plt.plot(ablation_history_df["val_loss"], label="No Dropout Validation Loss")
plt.title("Ablation Study: Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# Model Comparison

This table is the key result table for the report. It compares architecture changes, optimizer changes, ablation impact, F1-score, parameter count, and training time.

In [ ]:
final_comparison_df = pd.DataFrame(results)
final_comparison_df = final_comparison_df.sort_values(by="Test Accuracy", ascending=False).reset_index(drop=True)

display(final_comparison_df)

final_comparison_path = OUTPUT_DIR / "day3_to_day5_model_comparison.csv"
final_comparison_df.to_csv(final_comparison_path, index=False)

print("Saved final comparison to:", final_comparison_path)

plot_metric_comparison(final_comparison_df, "Test Accuracy", "Final Model Comparison: Test Accuracy")
plot_metric_comparison(final_comparison_df, "Macro F1", "Final Model Comparison: Macro F1")
plot_metric_comparison(final_comparison_df, "Training Time (minutes)", "Final Model Comparison: Training Time")
plot_metric_comparison(final_comparison_df, "Parameters", "Final Model Comparison: Parameter Count")

## Report-Ready Interpretation Template

After running the notebook, update the bracketed values with your actual results.

**Baseline CNN:** The baseline CNN established the reference performance using three convolutional layers, three pooling layers, and three fully connected layers. Its purpose was not necessarily to achieve the highest accuracy, but to provide a fair benchmark for deeper and regularized models.

**Deeper CNN:** The deeper CNN increased representational capacity using six convolutional layers. Batch Normalization stabilized learning, Dropout reduced overfitting, and L2 regularization penalized overly large weights. The model was compared with the baseline using accuracy, loss, F1-score, parameter count, and training time.

**Optimizer comparison:** Adam and SGD were compared on the same deeper CNN architecture. Adam is expected to converge faster because it adapts learning rates for individual parameters. SGD with momentum may sometimes generalize well, but it usually requires more careful learning-rate tuning.

**Ablation study:** Removing Dropout tested whether Dropout was contributing to generalization. If the no-Dropout model shows higher training accuracy but weaker validation or test performance, this indicates overfitting and supports the use of Dropout in the final architecture.

**Conclusion:** The best scratch-trained model should be selected based on validation/test accuracy, F1-score, confusion matrix behavior, and training efficiency—not accuracy alone.

# Day 6: Transfer Learning

This section uses the 224×224 datasets already created earlier:

- `train_ds_224`
- `val_ds_224`
- `test_ds_224`

The main transfer-learning model is **MobileNetV2** because it is efficient in Google Colab and performs strongly for image classification. **ResNet50** is included as an optional comparison model, but it is disabled by default to reduce runtime.

Transfer learning is completed in two stages:

1. **Feature extraction:** freeze the ImageNet convolutional base and train only the custom classification head.
2. **Fine-tuning:** unfreeze the final layers of the convolutional base and train using a very low learning rate.

In [ ]:
# ============================================================
# DAY 6: TRANSFER LEARNING SETUP
# ============================================================

from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess_input

required_day6_variables = [
    "train_ds_224", "val_ds_224", "test_ds_224",
    "CLASS_NAMES", "NUM_CLASSES", "OUTPUT_DIR",
    "results", "get_preprocessing_layers", "train_model",
    "evaluate_model", "plot_history", "plot_sample_predictions",
    "save_model", "add_result", "plot_metric_comparison"
]

missing_variables = [name for name in required_day6_variables if name not in globals()]

if missing_variables:
    raise NameError(
        "Run the earlier notebook cells before Day 6. Missing variables: " + ", ".join(missing_variables)
    )

print("Day 6 variables verified.")
print("Classes:", CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)

for images, labels in train_ds_224.take(1):
    print("Transfer learning image batch:", images.shape)
    print("Transfer learning label batch:", labels.shape)
    print("Pixel range before application preprocessing:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

## Transfer-Learning Configuration

The epoch counts are moderate to keep the full notebook practical in Colab. Early stopping and learning-rate reduction are handled by the existing `train_model()` helper.

In [ ]:
# ============================================================
# TRANSFER LEARNING CONFIGURATION
# ============================================================

TRANSFER_FEATURE_EPOCHS = 8
TRANSFER_FINE_TUNE_EPOCHS = 8
MOBILENET_FINE_TUNE_LAST_N = 40

# ResNet50 is slower. Keep False unless you have enough GPU time.
RUN_RESNET50 = True
RESNET50_FEATURE_EPOCHS = 6
RESNET50_FINE_TUNE_EPOCHS = 5
RESNET50_FINE_TUNE_LAST_N = 30

print("MobileNetV2 feature extraction epochs:", TRANSFER_FEATURE_EPOCHS)
print("MobileNetV2 fine-tuning epochs:", TRANSFER_FINE_TUNE_EPOCHS)
print("MobileNetV2 fine-tune last N layers:", MOBILENET_FINE_TUNE_LAST_N)
print("Run optional ResNet50:", RUN_RESNET50)

## Transfer-Learning Helper Functions

These functions reuse the earlier training and evaluation helpers. The input images remain in the original 0–255 range, and model-specific preprocessing is placed inside each transfer-learning model.

For MobileNetV2, pixels are scaled to approximately -1 to 1. For ResNet50, the official preprocessing function is applied in a `Lambda` layer.

In [ ]:
# ============================================================
# TRANSFER LEARNING HELPER FUNCTIONS
# ============================================================

def upsert_result(results, model, metrics, training_time_seconds):
    """Add or replace a result row so rerunning a cell does not duplicate rows."""
    model_name = metrics["model"]
    results = [row for row in results if row["Model"] != model_name]
    return add_result(results, model, metrics, training_time_seconds)


def build_mobilenetv2_model(input_shape=(224, 224, 3), num_classes=NUM_CLASSES):
    augmentation_layer, _ = get_preprocessing_layers()

    raw_base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )

    # Wrap the base model with a stable name so it can be retrieved for fine-tuning.
    base_model = keras.Model(
        inputs=raw_base_model.input,
        outputs=raw_base_model.output,
        name="MobileNetV2_base"
    )
    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = augmentation_layer(inputs)
    x = layers.Rescaling(1./127.5, offset=-1, name="mobilenetv2_preprocessing")(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dropout(0.30, name="classification_dropout_1")(x)
    x = layers.Dense(128, activation="relu", name="classification_dense_1")(x)
    x = layers.Dropout(0.20, name="classification_dropout_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="scene_classifier")(x)

    model = keras.Model(inputs, outputs, name="MobileNetV2_Transfer_Learning")
    return model, base_model


def build_resnet50_model(input_shape=(224, 224, 3), num_classes=NUM_CLASSES):
    augmentation_layer, _ = get_preprocessing_layers()

    raw_base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )

    base_model = keras.Model(
        inputs=raw_base_model.input,
        outputs=raw_base_model.output,
        name="ResNet50_base"
    )
    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = augmentation_layer(inputs)
    x = layers.Lambda(resnet50_preprocess_input, name="resnet50_preprocessing")(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dropout(0.35, name="classification_dropout_1")(x)
    x = layers.Dense(128, activation="relu", name="classification_dense_1")(x)
    x = layers.Dropout(0.25, name="classification_dropout_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="scene_classifier")(x)

    model = keras.Model(inputs, outputs, name="ResNet50_Transfer_Learning")
    return model, base_model


def prepare_for_fine_tuning(base_model, fine_tune_last_n=40):
    """Unfreeze only the last part of the convolutional base.

    Batch Normalization layers remain frozen for stable fine-tuning.
    """
    base_model.trainable = True

    if fine_tune_last_n is not None:
        for layer in base_model.layers[:-fine_tune_last_n]:
            layer.trainable = False

    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    trainable_layers = sum(1 for layer in base_model.layers if layer.trainable)
    frozen_layers = sum(1 for layer in base_model.layers if not layer.trainable)

    print("Trainable base layers:", trainable_layers)
    print("Frozen base layers:", frozen_layers)

    return base_model


def plot_transfer_learning_history(feature_history_df, fine_tune_history_df, model_name):
    feature_epochs = len(feature_history_df)

    combined_accuracy = list(feature_history_df["accuracy"]) + list(fine_tune_history_df["accuracy"])
    combined_val_accuracy = list(feature_history_df["val_accuracy"]) + list(fine_tune_history_df["val_accuracy"])
    combined_loss = list(feature_history_df["loss"]) + list(fine_tune_history_df["loss"])
    combined_val_loss = list(feature_history_df["val_loss"]) + list(fine_tune_history_df["val_loss"])
    epochs = range(1, len(combined_accuracy) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, combined_accuracy, label="Training Accuracy")
    plt.plot(epochs, combined_val_accuracy, label="Validation Accuracy")
    plt.axvline(feature_epochs, linestyle="--", label="Fine-tuning starts")
    plt.title(f"{model_name}: Feature Extraction + Fine-Tuning Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, combined_loss, label="Training Loss")
    plt.plot(epochs, combined_val_loss, label="Validation Loss")
    plt.axvline(feature_epochs, linestyle="--", label="Fine-tuning starts")
    plt.title(f"{model_name}: Feature Extraction + Fine-Tuning Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


## MobileNetV2: Feature Extraction

Only the new classification head is trained. The MobileNetV2 base remains frozen, which makes this stage fast and reduces the risk of overfitting.

In [ ]:
# ============================================================
# MOBILENETV2: FEATURE EXTRACTION MODEL
# ============================================================

mobilenet_model, mobilenet_base = build_mobilenetv2_model()
mobilenet_model.summary()

In [ ]:
# Retrieving features if exists in drive

mobilenet_feature_final = OUTPUT_DIR / "mobilenetv2_feature_extraction_final.keras"
mobilenet_feature_checkpoint = OUTPUT_DIR / "mobilenetv2_feature_extraction_best.keras"

if mobilenet_feature_final.exists():
    print("Feature extraction already done. Loading from Drive...")
    mobilenet_model = keras.models.load_model(mobilenet_feature_final)
    mobilenet_feature_history_df = pd.read_csv(
        OUTPUT_DIR / "mobilenetv2_feature_extraction_history.csv"
    )
    mobilenet_feature_training_time = 0
    skip_feature_extraction = True

elif mobilenet_feature_checkpoint.exists():
    print("Checkpoint found. Resuming from best saved epoch...")
    mobilenet_model = keras.models.load_model(mobilenet_feature_checkpoint)
    skip_feature_extraction = False

else:
    print("No checkpoint found. Building MobileNetV2 from scratch...")
    mobilenet_model, mobilenet_base = build_mobilenetv2_model()
    skip_feature_extraction = False

mobilenet_model.summary()

In [ ]:
# ============================================================
# TRAIN MOBILENETV2 FEATURE EXTRACTION MODEL
# ============================================================

# If feature extraction completed avoids the repeated feature extraction process

if not skip_feature_extraction:
    mobilenet_model, mobilenet_feature_history_df, mobilenet_feature_training_time = train_model(
        model=mobilenet_model,
        train_ds=train_ds_224,
        val_ds=val_ds_224,
        model_name="mobilenetv2_feature_extraction",
        epochs=TRANSFER_FEATURE_EPOCHS,
        optimizer_name="adam",
        learning_rate=0.001,
        patience=4
    )
    print("MobileNetV2 feature extraction training time minutes:",
          round(mobilenet_feature_training_time / 60, 2))

plot_history(mobilenet_feature_history_df, "MobileNetV2 Feature Extraction")
print("Feature extraction stage complete")

In [ ]:
# ============================================================
# EVALUATE MOBILENETV2 FEATURE EXTRACTION MODEL
# ============================================================

mobilenet_feature_metrics, mobilenet_feature_report_df, mobilenet_feature_cm = evaluate_model(
    mobilenet_model,
    test_ds_224,
    "mobilenetv2_feature_extraction"
)

save_model(mobilenet_model, "mobilenetv2_feature_extraction_final")
results = upsert_result(results, mobilenet_model, mobilenet_feature_metrics, mobilenet_feature_training_time)

## MobileNetV2: Fine-Tuning

The final layers of MobileNetV2 are unfrozen and trained using a low learning rate. This adapts pretrained visual features to the scene classification task without overwriting useful ImageNet representations.

In [ ]:
# ============================================================
# PREPARE MOBILENETV2 FOR FINE-TUNING
# ============================================================

mobilenet_finetune_final = OUTPUT_DIR / "mobilenetv2_finetuned_final.keras"
mobilenet_finetune_checkpoint = OUTPUT_DIR / "mobilenetv2_finetuned_best.keras"

# Checking if fine tuning was done prior else fine tuning from the beginning
if mobilenet_finetune_final.exists():
    print("Fine tuning already done. Loading from Drive...")
    mobilenet_model = keras.models.load_model(mobilenet_finetune_final)
    mobilenet_finetune_history_df = pd.read_csv(
        OUTPUT_DIR / "mobilenetv2_finetuned_history.csv"
    )
    mobilenet_finetune_training_time = 0
    skip_fine_tuning = True

elif mobilenet_finetune_checkpoint.exists():
    print("Fine tune checkpoint found. Resuming...")
    mobilenet_model = keras.models.load_model(mobilenet_finetune_checkpoint)
    skip_fine_tuning = False

else:
    print("Preparing for fine tuning...")
    # REASON: mobilenet_base may not exist in memory after disconnect
    # so we retrieve it directly from inside the loaded model
    mobilenet_base = mobilenet_model.get_layer("MobileNetV2_base")
    mobilenet_base = prepare_for_fine_tuning(
        mobilenet_base,
        fine_tune_last_n=MOBILENET_FINE_TUNE_LAST_N
    )
    skip_fine_tuning = False

mobilenet_model.summary()

In [ ]:
# ============================================================
# TRAIN MOBILENETV2 FINE-TUNED MODEL
# ============================================================

if not skip_fine_tuning:
    mobilenet_model, mobilenet_finetune_history_df, mobilenet_finetune_training_time = train_model(
        model=mobilenet_model,
        train_ds=train_ds_224,
        val_ds=val_ds_224,
        model_name="mobilenetv2_finetuned",
        epochs=TRANSFER_FINE_TUNE_EPOCHS,
        optimizer_name="adam",
        learning_rate=1e-5,
        patience=4
    )
    print("Fine tuning time minutes:",
          round(mobilenet_finetune_training_time / 60, 2))

mobilenet_total_training_time = (
    mobilenet_feature_training_time + mobilenet_finetune_training_time
)
print("Total MobileNetV2 time minutes:",
      round(mobilenet_total_training_time / 60, 2))

plot_history(mobilenet_finetune_history_df, "MobileNetV2 Fine Tuning")
plot_transfer_learning_history(
    mobilenet_feature_history_df,
    mobilenet_finetune_history_df,
    "MobileNetV2"
)
print("Fine tuning complete")

In [ ]:
# ============================================================
# EVALUATE MOBILENETV2 FINE-TUNED MODEL
# ============================================================

mobilenet_finetune_metrics, mobilenet_finetune_report_df, mobilenet_finetune_cm = evaluate_model(
    mobilenet_model,
    test_ds_224,
    "mobilenetv2_finetuned"
)

plot_sample_predictions(mobilenet_model, test_ds_224, "MobileNetV2 Fine-Tuned")
save_model(mobilenet_model, "mobilenetv2_finetuned_final")

# Use total time because the fine-tuned model depends on both training stages.
results = upsert_result(results, mobilenet_model, mobilenet_finetune_metrics, mobilenet_total_training_time)

## Optional: ResNet50 Transfer Learning

This section is disabled by default because ResNet50 is heavier than MobileNetV2. It is provided as an additional comparison if GPU time allows.

To run it, set:

```python
RUN_RESNET50 = True
```

In [ ]:
# ============================================================
# OPTIONAL RESNET50 TRANSFER LEARNING
# ============================================================

if RUN_RESNET50:
    resnet_model, resnet_base = build_resnet50_model()
    resnet_model.summary()

    resnet_model, resnet_feature_history_df, resnet_feature_training_time = train_model(
        model=resnet_model,
        train_ds=train_ds_224,
        val_ds=val_ds_224,
        model_name="resnet50_feature_extraction",
        epochs=RESNET50_FEATURE_EPOCHS,
        optimizer_name="adam",
        learning_rate=0.001,
        patience=3
    )

    plot_history(resnet_feature_history_df, "ResNet50 Feature Extraction")

    resnet_feature_metrics, resnet_feature_report_df, resnet_feature_cm = evaluate_model(
        resnet_model,
        test_ds_224,
        "resnet50_feature_extraction"
    )

    save_model(resnet_model, "resnet50_feature_extraction_final")
    results = upsert_result(results, resnet_model, resnet_feature_metrics, resnet_feature_training_time)

    resnet_base = prepare_for_fine_tuning(
        resnet_base,
        fine_tune_last_n=RESNET50_FINE_TUNE_LAST_N
    )

    resnet_model, resnet_finetune_history_df, resnet_finetune_training_time = train_model(
        model=resnet_model,
        train_ds=train_ds_224,
        val_ds=val_ds_224,
        model_name="resnet50_finetuned",
        epochs=RESNET50_FINE_TUNE_EPOCHS,
        optimizer_name="adam",
        learning_rate=1e-5,
        patience=3
    )

    resnet_total_training_time = resnet_feature_training_time + resnet_finetune_training_time

    plot_history(resnet_finetune_history_df, "ResNet50 Fine-Tuning")
    plot_transfer_learning_history(
        resnet_feature_history_df,
        resnet_finetune_history_df,
        "ResNet50"
    )

    resnet_finetune_metrics, resnet_finetune_report_df, resnet_finetune_cm = evaluate_model(
        resnet_model,
        test_ds_224,
        "resnet50_finetuned"
    )

    save_model(resnet_model, "resnet50_finetuned_final")
    results = upsert_result(results, resnet_model, resnet_finetune_metrics, resnet_total_training_time)

else:
    print("ResNet50 skipped to reduce runtime. Set RUN_RESNET50 = True if GPU time allows.")

## Day 6 Transfer-Learning Comparison

This table adds transfer-learning models to the scratch CNN experiments from Days 3–5.

In [ ]:
# ============================================================
# DAY 6 MODEL COMPARISON
# ============================================================

day6_comparison_df = pd.DataFrame(results)
day6_comparison_df = day6_comparison_df.drop_duplicates(subset=["Model"], keep="last")
day6_comparison_df = day6_comparison_df.sort_values(by="Test Accuracy", ascending=False).reset_index(drop=True)

display(day6_comparison_df)

day6_comparison_path = OUTPUT_DIR / "day6_transfer_learning_comparison.csv"
day6_comparison_df.to_csv(day6_comparison_path, index=False)

print("Saved Day 6 comparison to:", day6_comparison_path)

plot_metric_comparison(day6_comparison_df, "Test Accuracy", "Model Comparison Including Transfer Learning: Test Accuracy")
plot_metric_comparison(day6_comparison_df, "Macro F1", "Model Comparison Including Transfer Learning: Macro F1")
plot_metric_comparison(day6_comparison_df, "Training Time (minutes)", "Model Comparison Including Transfer Learning: Training Time")

# Day 7: Final Results, Error Analysis, and Report-Ready Outputs

Day 7 converts the modelling work into final deliverables for the report and viva. No EDA is repeated here.

This section produces:

1. Final model comparison table.
2. Final model ranking charts.
3. Best model identification.
4. Misclassification/error analysis.
5. A report-ready summary saved as a text file.
6. Viva preparation notes.

In [ ]:
# ============================================================
# DAY 7: FINAL COMPARISON TABLE
# ============================================================

final_all_models_df = pd.DataFrame(results)
final_all_models_df = final_all_models_df.drop_duplicates(subset=["Model"], keep="last")
final_all_models_df = final_all_models_df.sort_values(by="Test Accuracy", ascending=False).reset_index(drop=True)

final_all_models_display_df = final_all_models_df.copy()
for column in ["Test Accuracy", "Test Loss", "Macro F1", "Weighted F1", "Training Time (minutes)"]:
    final_all_models_display_df[column] = final_all_models_display_df[column].round(4)

final_all_models_display_df["Parameters"] = final_all_models_display_df["Parameters"].astype(int)

display(final_all_models_display_df)

final_all_models_path = OUTPUT_DIR / "final_all_models_comparison_day2_to_day7.csv"
final_all_models_display_df.to_csv(final_all_models_path, index=False)

print("Saved final all-model comparison to:", final_all_models_path)

In [ ]:
# ============================================================
# DAY 7: FINAL COMPARISON VISUALS
# ============================================================

plot_metric_comparison(final_all_models_df, "Test Accuracy", "Final All-Model Comparison: Test Accuracy")
plot_metric_comparison(final_all_models_df, "Macro F1", "Final All-Model Comparison: Macro F1")
plot_metric_comparison(final_all_models_df, "Training Time (minutes)", "Final All-Model Comparison: Training Time")
plot_metric_comparison(final_all_models_df, "Parameters", "Final All-Model Comparison: Parameter Count")

## Best Model Selection

The final model is selected primarily by **test accuracy** and **macro F1-score**. Macro F1 is useful because it gives equal importance to all classes.

In [ ]:
# ============================================================
# DAY 7: BEST MODEL IDENTIFICATION
# ============================================================

best_model_row = final_all_models_df.iloc[0]
best_model_name = best_model_row["Model"]

print("Best model based on test accuracy:")
print("Model:", best_model_name)
print("Test accuracy:", round(best_model_row["Test Accuracy"], 4))
print("Macro F1:", round(best_model_row["Macro F1"], 4))
print("Weighted F1:", round(best_model_row["Weighted F1"], 4))
print("Training time minutes:", round(best_model_row["Training Time (minutes)"], 2))
print("Parameter count:", int(best_model_row["Parameters"]))

trained_models = {}
if "baseline_model" in globals():
    trained_models["baseline_cnn"] = baseline_model
if "deeper_model" in globals():
    trained_models["deeper_cnn_adam"] = deeper_model
if "sgd_model" in globals():
    trained_models["deeper_cnn_sgd"] = sgd_model
if "ablation_model" in globals():
    trained_models["ablation_no_dropout"] = ablation_model
if "mobilenet_model" in globals():
    trained_models["mobilenetv2_finetuned"] = mobilenet_model
if "resnet_model" in globals():
    trained_models["resnet50_finetuned"] = resnet_model
    trained_models["resnet50_feature_extraction"] = resnet_model


def load_saved_model_if_available(model_name):
    model_path = OUTPUT_DIR / f"{model_name}_final.keras"

    if not model_path.exists():
        return None

    try:
        return keras.models.load_model(model_path)
    except TypeError:
        return keras.models.load_model(model_path, safe_mode=False)
    except Exception as error:
        print(f"Could not load saved model for {model_name}: {error}")
        return None

best_model = trained_models.get(best_model_name)

# The MobileNetV2 feature-extraction model is saved before fine-tuning.
# If it is the best row, load the saved feature-extraction model.
if best_model_name == "mobilenetv2_feature_extraction":
    best_model = load_saved_model_if_available("mobilenetv2_feature_extraction")

if best_model is None:
    best_model = load_saved_model_if_available(best_model_name)

if best_model is None:
    print("Best model object was not found. Re-run the corresponding training cell before error analysis.")
else:
    print("Best model ready for error analysis:", best_model.name)

## Error Analysis

This section displays high-confidence incorrect predictions. These examples help explain model limitations, especially between visually similar classes such as glacier, mountain, sea, buildings, and street.

In [ ]:
# ============================================================
# DAY 7: ERROR ANALYSIS HELPERS
# ============================================================

def prediction_details(model, dataset, paths, model_name):
    true_labels = []
    predicted_labels = []
    confidences = []

    for images, labels in dataset:
        probabilities = model.predict(images, verbose=0)
        true_batch = np.argmax(labels.numpy(), axis=1)
        pred_batch = np.argmax(probabilities, axis=1)
        confidence_batch = np.max(probabilities, axis=1)

        true_labels.extend(true_batch)
        predicted_labels.extend(pred_batch)
        confidences.extend(confidence_batch)

    details_df = pd.DataFrame({
        "path": paths[:len(true_labels)],
        "true_index": true_labels,
        "predicted_index": predicted_labels,
        "confidence": confidences
    })

    details_df["true_class"] = details_df["true_index"].map(index_to_class)
    details_df["predicted_class"] = details_df["predicted_index"].map(index_to_class)
    details_df["correct"] = details_df["true_index"] == details_df["predicted_index"]
    details_df["model"] = model_name

    return details_df


def plot_misclassified_examples(details_df, title="Misclassified Examples", number=9):
    wrong_df = details_df[details_df["correct"] == False].copy()

    if len(wrong_df) == 0:
        print("No misclassified examples found.")
        return

    wrong_df = wrong_df.sort_values(by="confidence", ascending=False).head(number)

    rows = int(np.ceil(number / 3))
    plt.figure(figsize=(12, rows * 4))

    for i, (_, row) in enumerate(wrong_df.iterrows()):
        image = Image.open(row["path"]).convert("RGB")

        plt.subplot(rows, 3, i + 1)
        plt.imshow(image)
        plt.title(
            f"True: {row['true_class']}\n"
            f"Pred: {row['predicted_class']} ({row['confidence']*100:.1f}%)"
        )
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# DAY 7: RUN ERROR ANALYSIS ON BEST MODEL
# ============================================================

if best_model is not None:
    if best_model.input_shape[1] == 224:
        error_analysis_dataset = test_ds_224
    else:
        error_analysis_dataset = test_ds_150

    error_details_df = prediction_details(
        model=best_model,
        dataset=error_analysis_dataset,
        paths=test_paths,
        model_name=best_model_name
    )

    display(error_details_df.head())

    incorrect_df = error_details_df[error_details_df["correct"] == False].copy()
    print("Total test examples:", len(error_details_df))
    print("Correct predictions:", int(error_details_df["correct"].sum()))
    print("Incorrect predictions:", len(incorrect_df))
    print("Error rate:", round(len(incorrect_df) / len(error_details_df), 4))

    display(
        incorrect_df
        .sort_values(by="confidence", ascending=False)
        .head(10)[["true_class", "predicted_class", "confidence", "path"]]
    )

    error_details_path = OUTPUT_DIR / "final_model_error_analysis.csv"
    error_details_df.to_csv(error_details_path, index=False)
    print("Saved error analysis to:", error_details_path)

    plot_misclassified_examples(
        error_details_df,
        title=f"{best_model_name}: High-Confidence Misclassified Examples",
        number=9
    )
else:
    print("Skipping error analysis because best_model is not available.")

## Report-Ready Final Summary

Run the next cell after all selected models have finished training. It generates a concise summary that can be adapted for the final report conclusion.

In [ ]:
# ============================================================
# DAY 7: SAVE REPORT-READY SUMMARY
# ============================================================

def get_accuracy_text(model_name):
    row = final_all_models_df[final_all_models_df["Model"] == model_name]
    if len(row) == 0:
        return "not available"
    return f"{float(row.iloc[0]['Test Accuracy']):.4f} test accuracy"

baseline_text = get_accuracy_text("baseline_cnn")
deep_text = get_accuracy_text("deeper_cnn_adam")
sgd_text = get_accuracy_text("deeper_cnn_sgd")
ablation_text = get_accuracy_text("ablation_no_dropout")
mobilenet_feature_text = get_accuracy_text("mobilenetv2_feature_extraction")
mobilenet_finetune_text = get_accuracy_text("mobilenetv2_finetuned")

summary_lines = []
summary_lines.append("FINAL MODELLING SUMMARY")
summary_lines.append("=" * 55)
summary_lines.append(f"Number of classes: {NUM_CLASSES}")
summary_lines.append(f"Classes: {', '.join(CLASS_NAMES)}")
summary_lines.append(f"Training samples: {len(train_paths)}")
summary_lines.append(f"Validation samples: {len(val_paths)}")
summary_lines.append(f"Testing samples: {len(test_paths)}")
summary_lines.append("")
summary_lines.append("Completed experiments:")
summary_lines.append(f"1. Baseline CNN from scratch: {baseline_text}")
summary_lines.append(f"2. Deeper CNN with regularization: {deep_text}")
summary_lines.append(f"3. Deeper CNN with SGD optimizer: {sgd_text}")
summary_lines.append(f"4. Ablation without Dropout: {ablation_text}")
summary_lines.append(f"5. MobileNetV2 feature extraction: {mobilenet_feature_text}")
summary_lines.append(f"6. MobileNetV2 fine-tuning: {mobilenet_finetune_text}")
if "resnet_model" in globals():
    summary_lines.append("7. Optional ResNet50 comparison was also executed.")
summary_lines.append("")
summary_lines.append("Best model according to test accuracy:")
summary_lines.append(f"Model: {best_model_name}")
summary_lines.append(f"Test accuracy: {float(best_model_row['Test Accuracy']):.4f}")
summary_lines.append(f"Macro F1-score: {float(best_model_row['Macro F1']):.4f}")
summary_lines.append(f"Weighted F1-score: {float(best_model_row['Weighted F1']):.4f}")
summary_lines.append(f"Training time: {float(best_model_row['Training Time (minutes)']):.2f} minutes")
summary_lines.append(f"Parameters: {int(best_model_row['Parameters']):,}")
summary_lines.append("")
summary_lines.append("Interpretation guidance:")
summary_lines.append("- The baseline CNN provides a reference point for models trained from scratch.")
summary_lines.append("- The deeper CNN tests whether increased convolutional depth improves generalization.")
summary_lines.append("- Batch Normalization stabilizes training, Dropout reduces overfitting, and L2 regularization penalizes large weights.")
summary_lines.append("- Adam and SGD are compared to evaluate convergence speed and final performance.")
summary_lines.append("- The ablation study isolates the impact of Dropout by removing it from the deeper CNN.")
summary_lines.append("- MobileNetV2 uses pretrained ImageNet features and is expected to improve performance with less training time.")
summary_lines.append("- Fine-tuning adapts the final pretrained layers to the scene classification dataset using a low learning rate.")
summary_lines.append("")
summary_lines.append("Conclusion template:")
summary_lines.append("The experiments show the progression from a simple CNN baseline to a regularized deeper CNN and finally to transfer learning. The final selected model should be justified using test accuracy, macro F1-score, confusion matrix behaviour, training time, parameter count, and error analysis.")

report_summary = "\n".join(summary_lines)
print(report_summary)

summary_path = OUTPUT_DIR / "final_report_ready_model_summary.txt"
summary_path.write_text(report_summary)

print("Saved report-ready summary to:", summary_path)


## Viva Preparation Notes

Be ready to explain the following clearly:

1. Why the corrupt image was excluded from the pipeline.
2. Why scratch CNNs use 150×150 input while transfer-learning models use 224×224 input.
3. What convolution, pooling, ReLU, softmax, and categorical cross-entropy do.
4. Why the deeper CNN uses Batch Normalization, Dropout, and L2 regularization.
5. Why Adam and SGD can produce different convergence behaviour.
6. What the ablation study proves about Dropout.
7. Why MobileNetV2 is suitable for Colab and scene classification.
8. Why fine-tuning uses a very low learning rate.
9. How the confusion matrix reveals which scene classes are visually confused.
10. Why the final selected model is best according to accuracy, F1-score, and computational cost.

# Notebook Completion Checklist

After running the notebook, confirm that the following files are saved in `OUTPUT_DIR`:

- `clean_metadata.csv`
- model files ending in `.keras`
- history CSV files for each model
- classification report CSV files
- `final_all_models_comparison_day2_to_day7.csv`
- `final_model_error_analysis.csv`
- `final_report_ready_model_summary.txt`

The notebook is now complete for the vision modelling workflow, excluding EDA.

In [ ]:
# ============================================================
# FINAL NOTEBOOK COMPLETION CHECK
# ============================================================

print("========== NOTEBOOK COMPLETION CHECK ==========")
print("Day 2: Clean data pipeline completed")
print("Day 3: Baseline CNN completed")
print("Day 4: Deeper CNN with regularization completed")
print("Day 5: Optimizer comparison and ablation study completed")
print("Day 6: MobileNetV2 transfer learning completed")
print("Day 7: Final comparison, error analysis, and report summary completed")
print("")
print("Main output files are saved in:", OUTPUT_DIR)
print("Final comparison CSV:", final_all_models_path)
print("Final report summary:", summary_path)